In [77]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [434]:
import json
import pathlib
import dotenv
import pickle
import sys
import csv
import os
import pandas as pd
import numpy as np
from typing import Dict, List
from ast import literal_eval
from datetime import datetime
from pydantic import BaseModel, Field, ValidationError, parse_obj_as

current_dir = pathlib.Path().parent.resolve()
sys.path.append(str(current_dir.parent))

from loguru import logger
from datahub_integrations.gen_ai.bedrock import BedrockModel
from datahub_integrations.gen_ai.term_suggestion_v2 import get_term_recommendations
from datahub_integrations.gen_ai.description_v2 import extract_metadata_for_urn, transform_table_info_for_llm
from datahub_integrations.gen_ai.term_suggestion_v2_context import fetch_glossary_info, GlossaryUniverseConfig
from docs_generation.graph_helper import create_datahub_graph
from term_suggestion_v2.helper import write_llm_output_to_csv

TERM_SUGGESTION_GENERATION_MODEL: BedrockModel = parse_obj_as(
    BedrockModel,
    os.getenv(
        "TERM_SUGGESTION_GENERATION_BEDROCK_MODEL", BedrockModel.CLAUDE_3_HAIKU.value
    ),
)

urns_dict: Dict[str, List[str]] = json.loads(
    (current_dir / "test_urns.json").read_text()
)

CONFIDENCE_THRESHOLD = 8


In [408]:
def get_table_and_column_infos_dict(urns_dict):
    table_infos_dict = {}
    column_infos_dict = {}
    for instance, urns in urns_dict.items():
        graph_client = create_datahub_graph(instance)
        for table_urn in urns:
            print(table_urn)
            try:
                entity = graph_client.get_entity_semityped(table_urn)
                extracted_entity_info = extract_metadata_for_urn(entity, table_urn, graph_client)
                table_info, column_info = transform_table_info_for_llm(extracted_entity_info)
                table_infos_dict[table_urn] = table_info
                column_infos_dict[table_urn] = column_info
            except Exception as e:
                print(e)
                continue
    return table_infos_dict, column_infos_dict

In [409]:
dotenv.load_dotenv(current_dir / ".env")

True

In [414]:
prompt = (current_dir / "../../src/datahub_integrations/gen_ai/term_suggestion_prompt.txt").read_text()
prompt

'You are tasked with assigning the appropriate glossary terms from the list of glossary terms provided in the prompt.\nPlease follow the instructions and ensure your output adheres to the specified format.\n\n**Information Provided in the prompt**\n<Glossary Terms>\n{glossary_terms_info}\n</Glossary Terms>\n\nGlossary Terms is a dictionary of available glossary terms which can be assigned to tables and columns.\nGiven below is the structure of the provided dictionary of glossary terms:\n{{\n    "<term1_unique_identifier>": {{\n               "term_name": "<name of the glossary term>",\n                "term_description": "<description of the glossary term>",\n                "parent_node": {{\n                                 "name": "<parent node name>",\n                                 "description": "<description of the parent node>"\n                }}\n           }},\n    "<term2_unique_identifier>": {{\n               "term_name": "<name of the glossary term>",\n                

In [411]:
for instance in urns_dict.keys():
    print(instance)
    try:
        create_datahub_graph(instance)
    except:
        print("failed")
        continue

acryl
chime
notion
twilio
longtailcompanions


In [412]:
graph_client = create_datahub_graph("longtailcompanions")
glossary_config = GlossaryUniverseConfig(glossary_nodes=["urn:li:glossaryNode:PIITerms"])


glossary_info = fetch_glossary_info(
    graph_client=graph_client,
    universe=glossary_config
)

In [120]:
# urns_dict_ = {key: value for key, value in urns_dict.items() if key in ['longtailcompanions', 'acryl']}

In [413]:
CSV_PATH = f"all_urns_analysis/term_suggestion_output_{datetime.now().strftime('%m-%d-%Y_%H-%M-%S')}.csv"
PKL_PATH = f"{CSV_PATH.rsplit(".")[0]}.pkl"
PROMPT_PATH = f"{CSV_PATH.rsplit(".")[0]}_prompt.txt"

raw_llm_responses = []
parsed_llm_responses = []


for instance, urns in urns_dict.items():
    graph_client = create_datahub_graph(instance)
    for urn in urns:
        print(urn)
        try:
            table_terms, column_terms, raw_llm_response = get_term_recommendations(
                table_urn=urn,
                graph_client=graph_client,
                glossary_info=glossary_info
            )
#             column_terms = label_fake_terms(column_terms)
            raw_llm_responses.append([urn, instance, raw_llm_response])
            parsed_llm_responses.append((urn, instance, table_terms, column_terms))
        except Exception as e:
            logger.exception(f"Exception Occurred {e}")
            raw_llm_responses.append([urn, instance, raw_llm_response])
            parsed_llm_responses.append((urn, instance, None, None))
            continue


write_llm_output_to_csv(llm_response=parsed_llm_responses, csv_path=CSV_PATH)
with open(PKL_PATH, 'wb') as fp:
    pickle.dump(parsed_llm_responses, fp)
    fp.close()

urn:li:dataset:(urn:li:dataPlatform:snowflake,datahub_community.orbit_fivetran.member_in_workspace,PROD)


2024-09-05 20:40:49.903 | DEBUG    | datahub_integrations.gen_ai.bedrock:call_bedrock_llm:84 - LLM call took 20.78225588798523 seconds


urn:li:dataset:(urn:li:dataPlatform:snowflake,datahub_community.orbit.workspace_activity,PROD)


2024-09-05 20:41:12.943 | DEBUG    | datahub_integrations.gen_ai.bedrock:call_bedrock_llm:84 - LLM call took 20.038954734802246 seconds


urn:li:dataset:(urn:li:dataPlatform:snowflake,analytics.looker.zendesk_agent,PROD)


...

urn:li:dataset:(urn:li:dataPlatform:looker,long_tail_companions.view.pet_details,PROD)


2024-09-05 20:55:34.574 | DEBUG    | datahub_integrations.gen_ai.bedrock:call_bedrock_llm:84 - LLM call took 369.7775559425354 seconds


urn:li:dataset:(urn:li:dataPlatform:looker,long_tail_companions.explore.monthly_adoption_projection,PROD)


2024-09-05 20:56:06.635 | DEBUG    | datahub_integrations.gen_ai.bedrock:call_bedrock_llm:84 - LLM call took 22.602768659591675 seconds


csv file all_urns_analysis/term_suggestion_output_09-05-2024_20-40-27.csv created successfully


In [ ]:
# from datahub_integrations.gen_ai.term_suggestion_v2 import parse_llm_output
# from datahub_integrations.gen_ai.bedrock import call_bedrock_llm
# parse_llm_output(raw_llm_response)

In [ ]:
# raw_llm_response

In [415]:
with open(PROMPT_PATH, 'w') as fp:
    fp.write(prompt)
    fp.close()

In [75]:
# promtp_llm_output = call_bedrock_llm(
#     prompt=f"""
#             I am using LLM models to assign glossary terms to the columns of a table for which I have created one prompt
#             Task: To Modify the provided prompt to make it appropriate to use in amazon bedrock llm models 
#             <prompt>
#             {prompt}
#             <prompt>
#             """,
#     model = TERM_SUGGESTION_GENERATION_MODEL,
#     max_tokens=4000
# )

2024-09-05 17:44:06.285 | DEBUG    | datahub_integrations.gen_ai.bedrock:call_bedrock_llm:84 - LLM call took 9.406734228134155 seconds


In [76]:
# with open("new_prompt.txt", 'w') as fp:
#     fp.write(promtp_llm_output)
#     fp.close()

In [376]:
# CSV_PATH = f"all_urns_analysis/term_suggestion_output_09-05-2024_18-05-44.csv"
# PKL_PATH = f"{CSV_PATH.rsplit(".")[0]}.pkl"
# PROMPT_PATH = f"{CSV_PATH.rsplit(".")[0]}_prompt.txt"
# with open(PKL_PATH, 'rb') as fp:
#     parsed_llm_responses = pickle.load(fp)
#     fp.close()

In [435]:
df = pd.read_csv("column_labels.csv")
df.columns = df.iloc[0]
df = df.iloc[1:, 1:]
df.loc[:, "unique_keys"] = df.table_urn + "_" + df.col_name
df.loc[:, "unique_keys"] = df.unique_keys.astype(str)
df = df.replace([np.nan], [None])
df = df[df['table_urn'].isin([urn for value in urns_dict.values() for urn in value])]
df = df.reset_index(drop=True)
df.head()

,table_urn,col_name,col_description,sample_values,glossary_term,Alternate_Glossary_Or_Comment,Harshal’s comments,unique_keys
0,"urn:li:dataset:(urn:li:dataPlatform:snowflake,...",id,None,None,None,None,None,"urn:li:dataset:(urn:li:dataPlatform:snowflake,..."
1,"urn:li:dataset:(urn:li:dataPlatform:snowflake,...",deleted_at,None,None,None,None,None,"urn:li:dataset:(urn:li:dataPlatform:snowflake,..."
2,"urn:li:dataset:(urn:li:dataPlatform:snowflake,...",updated_at,None,None,None,None,None,"urn:li:dataset:(urn:li:dataPlatform:snowflake,..."
3,"urn:li:dataset:(urn:li:dataPlatform:snowflake,...",topics,None,None,None,None,None,"urn:li:dataset:(urn:li:dataPlatform:snowflake,..."
4,"urn:li:dataset:(urn:li:dataPlatform:snowflake,...",created,None,None,None,None,None,"urn:li:dataset:(urn:li:dataPlatform:snowflake,..."


In [436]:
def get_failed_response_table_urns(parsed_llm_responses):
    tables_with_parsing_error = []
    skipped_tables = []
    for item in parsed_llm_responses:
        urn = item[0]
        column_terms = item[3]
        if column_terms is None:
            tables_with_parsing_error.append(urn)
        elif len(column_terms)==0:
            skipped_tables.append(urn)
#             print(len(column_terms))
        else:
#             print(urn)
#             print(len(column_terms))
            continue
    return tables_with_parsing_error, skipped_tables
tables_with_parsing_error, skipped_tables  = get_failed_response_table_urns(parsed_llm_responses) 
print("tables_with_parsing_error:\n", tables_with_parsing_error)
print("skipped_tables:\n", skipped_tables)

tables_with_parsing_error:
 ['urn:li:dataset:(urn:li:dataPlatform:snowflake,production.core.dim_all_workspace_member,PROD)', 'urn:li:dataset:(urn:li:dataPlatform:snowflake,production.core.stripe_customer_daily,PROD)', 'urn:li:dataset:(urn:li:dataPlatform:presto,twilio.us1.CDAX.hive.public.scd1_zendesk_users,PROD)']
skipped_tables:
 []


In [437]:
def get_df_pred(parsed_llm_responses, confidence_threshold = CONFIDENCE_THRESHOLD):
    terms_assigned = {}
    for response in parsed_llm_responses:
#         print(response[0], response[1])
        instance, table_urn = response[1], response[0]
        column_terms = response[3]
        if column_terms is not None:
            for column_name,  terms in column_terms.items():
                unique_key = table_urn + "_" + column_name
                if terms is not None:
                    terms_list = [(term.name, term.confidence_score) for term in terms if not term.is_fake]
                else:
                    terms_list = []
                terms_assigned[unique_key] = terms_list
    df_pred = pd.DataFrame({"unique_keys": list(terms_assigned.keys()), "predicted_labels": list(terms_assigned.values())})
    df_pred.loc[:, "pred_max_score_term"] = df_pred["predicted_labels"].apply(
        lambda x: [
            term[0] for term in x if (term[1]==max([i[1] for i in x])) and (term[1] >= confidence_threshold)
        ]
    )
    return df_pred
df_pred = get_df_pred(parsed_llm_responses, CONFIDENCE_THRESHOLD)

In [438]:
print(len(df))
print(len(df_pred))

987
611


In [439]:
CONFIDENCE_THRESHOLD

8

In [440]:
df.head()

,table_urn,col_name,col_description,sample_values,glossary_term,Alternate_Glossary_Or_Comment,Harshal’s comments,unique_keys
0,"urn:li:dataset:(urn:li:dataPlatform:snowflake,...",id,None,None,None,None,None,"urn:li:dataset:(urn:li:dataPlatform:snowflake,..."
1,"urn:li:dataset:(urn:li:dataPlatform:snowflake,...",deleted_at,None,None,None,None,None,"urn:li:dataset:(urn:li:dataPlatform:snowflake,..."
2,"urn:li:dataset:(urn:li:dataPlatform:snowflake,...",updated_at,None,None,None,None,None,"urn:li:dataset:(urn:li:dataPlatform:snowflake,..."
3,"urn:li:dataset:(urn:li:dataPlatform:snowflake,...",topics,None,None,None,None,None,"urn:li:dataset:(urn:li:dataPlatform:snowflake,..."
4,"urn:li:dataset:(urn:li:dataPlatform:snowflake,...",created,None,None,None,None,None,"urn:li:dataset:(urn:li:dataPlatform:snowflake,..."


In [441]:
merged_df = pd.merge(df[~df.table_urn.isin(tables_with_parsing_error)], df_pred, on="unique_keys", how = 'left')
merged_df.loc[:, 'pred_max_score_term'] = merged_df['pred_max_score_term'].apply(lambda x: x if isinstance(x, list) else [])
merged_df.loc[:, 'predicted_labels'] = merged_df['predicted_labels'].apply(lambda x: x if isinstance(x, list) else [])
not_omitted_columns_df = merged_df[merged_df.unique_keys.isin(df_pred.unique_keys.tolist())]
omitted_columns_df = merged_df[~merged_df.unique_keys.isin(df_pred.unique_keys.tolist())]

In [442]:
len(merged_df)

620

In [443]:
# merged_df

In [444]:
prediction_results_df = merged_df

In [445]:
print(len(not_omitted_columns_df))
print(len(omitted_columns_df))

610
10


In [446]:
# TODO: KEEP HIGHEST CONFIDENCE:
def func_categorize(row):
    if row["glossary_term"] is None and len(row["pred_max_score_term"])==0:
        return "match-no_assignment"
    elif (
        (row["glossary_term"] in row["pred_max_score_term"])
        or 
        (row["Alternate_Glossary_Or_Comment"] in row["pred_max_score_term"])
    ):
        return "match-term_assigned"
    elif row["glossary_term"] is None and len(row["pred_max_score_term"]) != 0:
        return "mismatch-predicted_only"
    elif row["glossary_term"] is not None and len(row["pred_max_score_term"]) == 0:
        return "mismatch_actual_only"
    elif (
            (row["glossary_term"] not in row["pred_max_score_term"])
            and
            (row["Alternate_Glossary_Or_Comment"] not in row["pred_max_score_term"])
    ):
        return "mismatch"
    else:
        return "some_error"
    
def check_match(row):
#     print(row)
    if (
        (row["glossary_term"] is None and len(row["pred_max_score_term"])==0) or 
        row["glossary_term"] in row["pred_max_score_term"] or
        row["Alternate_Glossary_Or_Comment"] in row["pred_max_score_term"]
    ) :
        return 1
    else:
        return 0
prediction_results_df.loc[:, "match"] = prediction_results_df.apply(lambda x: check_match(x), axis = 1)
prediction_results_df.loc[:, "label_class"] = prediction_results_df.apply(lambda x: func_categorize(x), axis = 1)

In [447]:
prediction_results_df.label_class.value_counts()

label_class
match-no_assignment        432
mismatch-predicted_only    118
match-term_assigned         51
mismatch_actual_only        12
mismatch                     7
Name: count, dtype: int64

In [448]:
prediction_results_df[prediction_results_df.label_class=="mismatch-predicted_only"]

,table_urn,col_name,col_description,sample_values,glossary_term,Alternate_Glossary_Or_Comment,Harshal’s comments,unique_keys,predicted_labels,pred_max_score_term,match,label_class
0,"urn:li:dataset:(urn:li:dataPlatform:snowflake,...",id,None,None,None,None,None,"urn:li:dataset:(urn:li:dataPlatform:snowflake,...","[(DeviceID, 8.0)]",[DeviceID],0,mismatch-predicted_only
6,"urn:li:dataset:(urn:li:dataPlatform:snowflake,...",url,None,None,None,None,None,"urn:li:dataset:(urn:li:dataPlatform:snowflake,...","[(LinkedInProfile, 8.0)]",[LinkedInProfile],0,mismatch-predicted_only
8,"urn:li:dataset:(urn:li:dataPlatform:snowflake,...",github,None,None,None,None,None,"urn:li:dataset:(urn:li:dataPlatform:snowflake,...","[(LinkedInProfile, 8.0)]",[LinkedInProfile],0,mismatch-predicted_only
18,"urn:li:dataset:(urn:li:dataPlatform:snowflake,...",orbit_url,None,None,None,None,None,"urn:li:dataset:(urn:li:dataPlatform:snowflake,...","[(LinkedInProfile, 8.0)]",[LinkedInProfile],0,mismatch-predicted_only
21,"urn:li:dataset:(urn:li:dataPlatform:snowflake,...",location,None,None,None,None,None,"urn:li:dataset:(urn:li:dataPlatform:snowflake,...","[(CountryOfResidence, 8.0)]",[CountryOfResidence],0,mismatch-predicted_only
...,...,...,...,...,...,...,...,...,...,...,...,...
613,"urn:li:dataset:(urn:li:dataPlatform:looker,lon...",pet_details.updated_time,None,None,None,None,None,"urn:li:dataset:(urn:li:dataPlatform:looker,lon...","[(BirthDate, 8.0)]",[BirthDate],0,mismatch-predicted_only
614,"urn:li:dataset:(urn:li:dataPlatform:looker,lon...",pet_details.updated_week,None,None,None,None,None,"urn:li:dataset:(urn:li:dataPlatform:looker,lon...","[(BirthDate, 8.0)]",[BirthDate],0,mismatch-predicted_only
615,"urn:li:dataset:(urn:li:dataPlatform:looker,lon...",pet_details.updated_year,None,None,None,None,None,"urn:li:dataset:(urn:li:dataPlatform:looker,lon...","[(BirthDate, 8.0)]",[BirthDate],0,mismatch-predicted_only
616,"urn:li:dataset:(urn:li:dataPlatform:looker,lon...",pet_details.weeks_duration_in_status,None,None,None,None,None,"urn:li:dataset:(urn:li:dataPlatform:looker,lon...","[(Age, 8.0)]",[Age],0,mismatch-predicted_only


# Large vs small table analysis

In [137]:
table_infos_dict, column_infos_dict = get_table_and_column_infos_dict(urns_dict)

urn:li:dataset:(urn:li:dataPlatform:snowflake,datahub_community.orbit_fivetran.member_in_workspace,PROD)
urn:li:dataset:(urn:li:dataPlatform:snowflake,datahub_community.orbit.workspace_activity,PROD)
urn:li:dataset:(urn:li:dataPlatform:snowflake,analytics.looker.zendesk_agent,PROD)
urn:li:dataset:(urn:li:dataPlatform:looker,snowflake.view.penny_claim_created,PROD)
urn:li:dataset:(urn:li:dataPlatform:snowflake,analytics.omx.disputes_omxri,PROD)
urn:li:dataset:(urn:li:dataPlatform:snowflake,analytics.looker.user_socure_cip_dim,PROD)
urn:li:dataset:(urn:li:dataPlatform:snowflake,analytics.omx.five9_call_acd_sla,PROD)
urn:li:dataset:(urn:li:dataPlatform:snowflake,analytics.omx.five9_call_acd_sla_view,PROD)
urn:li:dataset:(urn:li:dataPlatform:snowflake,postgres_db.mms.external_card_transfers,PROD)
urn:li:dataset:(urn:li:dataPlatform:snowflake,edw_db.core.fct_settled_transaction,PROD)
urn:li:dataset:(urn:li:dataPlatform:snowflake,production.core.dim_all_workspace_member,PROD)
urn:li:dataset:

In [449]:
def get_table_fake_columns_data(column_terms, all_column_names):
    fake_columns_count = 0
    if column_terms is not None and not isinstance(column_terms, str):
        for key, values in column_terms.items():
            try:
                if key not in all_column_names:
                    fake_columns.append(key)
                    fake_columns_count+=1
                    
            except Exception as e:
                print(f"Exception!! {e}")
                continue
    return fake_columns_count
    
def get_table_fake_terms_data(column_terms):
    fake_terms_count = 0
    if column_terms is not None and not isinstance(column_terms, str):
        for key, values in column_terms.items():
            try:
                for value in values:
                    if value.is_fake is not None and value.is_fake:
                        fake_terms_count += 1
            except Exception as e:
                print(f"Exception!! {e}")
                continue
    return fake_terms_count

def get_classification_stats(table_urn, pred_df):
#     for table_urn in table_wise_analysis_dict.keys():
    df_temp = pred_df[pred_df.table_urn==table_urn]
    label_class_counts = dict(df_temp.label_class.value_counts())
    TP = label_class_counts.get('match-term_assigned', 0)
    TN = label_class_counts.get('match-no_assignment', 0)
    FP1 = label_class_counts.get('mismatch-predicted_only', 0)
    FP2 = label_class_counts.get('mismatch', 0)
    FN = label_class_counts.get('mismatch_actual_only', 0)
#     print(label_class_counts)
    print('TP: ', TP, ' FP1: ', FP1, " FP2: ", FP2, ' TN: ', TN, ' FN: ', FN)
    precision_term = TP/(TP + FP1 + FP2) if TP + FP1 + FP2 != 0 else np.nan
    recall_term = TP/(TP + FN + FP2) if TP + FN + FP2 !=0 else np.nan
    precision_none = TN/(TN + FN) if TN + FN !=0 else np.nan
    recall_none = TN/(TN + FP1) if TN + FP1 != 0 else np.nan
    return {
            'TP': TP,
            'TN': TN,
            'FP1': FP1,
            'FP2': FP2,
            'FN': FN,
            'recall_term': recall_term,
            'precision_term': precision_term,
            'recall_none': recall_none,
            'precision_none': precision_none
        }

# table_wise_analysis_dict = {}
# table_wise_fake_counts = []
def get_classification_report_df(parsed_llm_responses, pred_df, column_infos_dict):
    table_wise_analysis_dict = {}
    for i, response in enumerate(parsed_llm_responses):
        try:
            column_stats_dict = {}
            table_urn = response[0]
            column_terms = response[3]
            column_terms_character_length = len(str(column_infos_dict[table_urn]))
            if column_terms is not None:
                assigned_column_names = [column for column in column_terms.keys()]
                all_column_names = list(column_infos_dict[table_urn].keys())
                fake_columns_count = get_table_fake_columns_data(column_terms, all_column_names)
                column_stats_dict["fake_columns_count"] = fake_columns_count

                fake_terms_count = get_table_fake_terms_data(column_terms)
                column_stats_dict["fake_terms_count"] = fake_terms_count
                column_stats_dict["actual_column_count"] = len(all_column_names)
                column_stats_dict["skipped_columns_count"] = len(all_column_names) - len(assigned_column_names)
                column_stats_dict["column_terms_character_length"] = column_terms_character_length
                table_wise_analysis_dict[table_urn] = column_stats_dict
                table_wise_analysis_dict[table_urn].update(get_classification_stats(table_urn, pred_df))
            else:
                table_wise_analysis_dict[table_urn] = {"column_terms_character_length": column_terms_character_length}
        except Exception as e:
            print(e)
            continue
    classification_report_df = pd.DataFrame(table_wise_analysis_dict).transpose()
    classification_report_df.index.names=["table_urn"]
    classification_report_df = classification_report_df.reset_index()
    return classification_report_df
        
        
classification_report_df = get_classification_report_df(parsed_llm_responses, prediction_results_df, column_infos_dict)
precision_term_for_all_tables = classification_report_df.TP.sum()/(classification_report_df.TP.sum() + classification_report_df.FP1.sum() + classification_report_df.FP2.sum())
recall_term_for_all_tables = classification_report_df.TP.sum()/(classification_report_df.TP.sum() + classification_report_df.FN.sum() + classification_report_df.FP2.sum())
precision_none_for_all_tables = classification_report_df.TN.sum()/(classification_report_df.TN.sum() + classification_report_df.FN.sum())
recall_none_for_all_tables = classification_report_df.TN.sum()/(classification_report_df.TN.sum() + classification_report_df.FP1.sum())

TP:  8  FP1:  10  FP2:  0  TN:  24  FN:  0
TP:  0  FP1:  2  FP2:  0  TN:  21  FN:  1
TP:  4  FP1:  12  FP2:  1  TN:  22  FN:  1
TP:  2  FP1:  4  FP2:  0  TN:  4  FN:  0
TP:  2  FP1:  32  FP2:  1  TN:  46  FN:  0
TP:  1  FP1:  2  FP2:  0  TN:  19  FN:  0
TP:  4  FP1:  4  FP2:  0  TN:  63  FN:  0
TP:  4  FP1:  4  FP2:  0  TN:  63  FN:  0
TP:  2  FP1:  2  FP2:  0  TN:  23  FN:  0
TP:  3  FP1:  2  FP2:  0  TN:  57  FN:  3
TP:  1  FP1:  0  FP2:  0  TN:  10  FN:  0
'urn:li:dataset:(urn:li:dataPlatform:presto,twilio.us1.CDAX.hive.public.scd1_zendesk_users,PROD)'
TP:  0  FP1:  2  FP2:  1  TN:  2  FN:  0
TP:  0  FP1:  3  FP2:  0  TN:  3  FN:  0
TP:  4  FP1:  4  FP2:  1  TN:  5  FN:  0
TP:  1  FP1:  5  FP2:  0  TN:  6  FN:  4
TP:  8  FP1:  2  FP2:  0  TN:  6  FN:  0
TP:  0  FP1:  2  FP2:  0  TN:  3  FN:  0
TP:  1  FP1:  2  FP2:  0  TN:  3  FN:  0
TP:  0  FP1:  2  FP2:  0  TN:  18  FN:  1
TP:  0  FP1:  1  FP2:  1  TN:  13  FN:  0
TP:  2  FP1:  0  FP2:  1  TN:  13  FN:  2
TP:  4  FP1:  21  FP2:  1

In [450]:
classification_report_df[['fake_columns_count', 'fake_terms_count',
       'actual_column_count', 'skipped_columns_count', 'TP', 'TN', 'FP1', 'FP2', 'FN']].sum(axis=0)

fake_columns_count         0.0
fake_terms_count           3.0
actual_column_count      621.0
skipped_columns_count     10.0
TP                        51.0
TN                       432.0
FP1                      118.0
FP2                        7.0
FN                        12.0
dtype: float64

In [451]:
# instruction 1: Do not assign if nothing seems fit.
# instruction 2: Improving the model's understanding of the context.

In [452]:
# parsed_llm_responses

In [453]:
print('precision_term_for_all_tables: ', precision_term_for_all_tables)
print('recall_term_for_all_tables: ', recall_term_for_all_tables)
print('precision_none_for_all_tables: ', precision_none_for_all_tables)
print('recall_none_for_all_tables: ', recall_none_for_all_tables)

precision_term_for_all_tables:  0.2897727272727273
recall_term_for_all_tables:  0.7285714285714285
precision_none_for_all_tables:  0.972972972972973
recall_none_for_all_tables:  0.7854545454545454


     
 
     **actual**            **predicted**           **cateory_name**              
       None                    None                  match-no_assignment              TN
       None                    term                  mismatch-predicted_only          FP1
       term                    different term        mismatch                         FP2
       term                    term                  match-term_assigned              TP
       term                    None                  mismatch-actual_only             FN

    Recall of positive class:  TP/(TP + FN + FP2)
    Precision : TP/(TP + FP1 + FP2)

    Recal of negative class: TN/(TN + FP1)
    Precision of negative class: TN/(TN + FN)    
    

In [ ]:
no assignment - negative
assignment - positive

In [454]:
# final_stats = {
#     "precision_term_for_all_tables":  precision_term_for_all_tables,
#     "recall_term_for_all_tables":  recall_term_for_all_tables,
#     "precision_none_for_all_tables":  precision_none_for_all_tables,
#     "recall_none_for_all_tables":  recall_none_for_all_tables,
# }
# FINAL_STATS_PATH = f"{CSV_PATH.rsplit(".")[0]}_final_stats_threshold_{CONFIDENCE_THRESHOLD}.txt"
# with open(FINAL_STATS_PATH, 'w') as fp:
#     for key, value in final_stats.items():
#         fp.write(f"{key}: {value}")
#     fp.write(str(final_stats))
#     fp.write(str(dict(classification_report_df[['fake_columns_count', 'fake_terms_count',
#        'actual_column_count', 'skipped_columns_count', 'TP', 'TN', 'FP1', 'FP2', 'FN']].sum(axis=0))))
#     fp.close()

In [455]:
final_stats = dict(classification_report_df[['fake_columns_count', 'fake_terms_count',
       'actual_column_count', 'skipped_columns_count', 'TP', 'TN', 'FP1', 'FP2', 'FN']].sum(axis=0))
final_stats.update({
    "precision_term_for_all_tables":  precision_term_for_all_tables,
    "recall_term_for_all_tables":  recall_term_for_all_tables,
    "precision_none_for_all_tables":  precision_none_for_all_tables,
    "recall_none_for_all_tables":  recall_none_for_all_tables,
})
FINAL_STATS_PATH = f"{CSV_PATH.rsplit(".")[0]}_final_stats_threshold_{CONFIDENCE_THRESHOLD}.txt"
with open(FINAL_STATS_PATH, 'w') as fp:
    for key, value in final_stats.items():
        fp.write(f"{key}: {value}\n")
    fp.close()

In [456]:
classification_report_df.to_csv(f"{CSV_PATH.rsplit(".")[0]}_classification_report_threshold_{CONFIDENCE_THRESHOLD}.csv")

In [260]:
classification_report_df

,table_urn,fake_columns_count,fake_terms_count,actual_column_count,skipped_columns_count,column_terms_character_length,TP,TN,FP1,FP2,FN,recall_term,precision_term,recall_none,precision_none
0,"urn:li:dataset:(urn:li:dataPlatform:snowflake,...",0.0,0.0,42.0,0.0,4900.0,4.0,34.0,0.0,0.0,4.0,0.500000,1.000000,1.000000,0.894737
1,"urn:li:dataset:(urn:li:dataPlatform:snowflake,...",0.0,22.0,24.0,1.0,15983.0,0.0,22.0,1.0,0.0,1.0,0.000000,0.000000,0.956522,0.956522
2,"urn:li:dataset:(urn:li:dataPlatform:snowflake,...",0.0,0.0,40.0,27.0,14506.0,3.0,34.0,0.0,0.0,3.0,0.500000,1.000000,1.000000,0.918919
3,"urn:li:dataset:(urn:li:dataPlatform:looker,sno...",0.0,1.0,10.0,0.0,4980.0,2.0,8.0,0.0,0.0,0.0,1.000000,1.000000,1.000000,1.000000
4,"urn:li:dataset:(urn:li:dataPlatform:snowflake,...",0.0,0.0,82.0,75.0,32312.0,2.0,78.0,0.0,0.0,1.0,0.666667,1.000000,1.000000,0.987342
5,"urn:li:dataset:(urn:li:dataPlatform:snowflake,...",0.0,0.0,22.0,0.0,8421.0,1.0,20.0,1.0,0.0,0.0,1.000000,0.500000,0.952381,1.000000
6,"urn:li:dataset:(urn:li:dataPlatform:snowflake,...",NaN,NaN,NaN,NaN,26681.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,"urn:li:dataset:(urn:li:dataPlatform:snowflake,...",0.0,0.0,71.0,0.0,33814.0,4.0,65.0,2.0,0.0,0.0,1.000000,0.666667,0.970149,1.000000
8,"urn:li:dataset:(urn:li:dataPlatform:snowflake,...",0.0,6.0,27.0,0.0,7652.0,1.0,21.0,4.0,0.0,1.0,0.500000,0.200000,0.840000,0.954545
9,"urn:li:dataset:(urn:li:dataPlatform:snowflake,...",NaN,NaN,NaN,NaN,9162.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [112]:
prediction_results_df[prediction_results_df.label_class == 'mismatch_actual_only']

,table_urn,col_name,col_description,sample_values,glossary_term,Alternate_Glossary_Or_Comment,Harshal’s comments,unique_keys,predicted_labels,pred_max_score_term,match,label_class
24,"urn:li:dataset:(urn:li:dataPlatform:snowflake,...",pronouns,None,None,GenderIdentity,None,None,"urn:li:dataset:(urn:li:dataPlatform:snowflake,...","[(GenderIdentity, 8.0)]",[],0,mismatch_actual_only
57,"urn:li:dataset:(urn:li:dataPlatform:snowflake,...",g_merged_by,None,"['noggi', 'hsheth2', 'anshbansal', 'hsheth2', ...",Username,None,The sample data should make it clear that this...,"urn:li:dataset:(urn:li:dataPlatform:snowflake,...",[],[],0,mismatch_actual_only
136,"urn:li:dataset:(urn:li:dataPlatform:snowflake,...",user_id,None,None,Username,None,None,"urn:li:dataset:(urn:li:dataPlatform:snowflake,...","[(Username, 8.0)]",[],0,mismatch_actual_only
323,"urn:li:dataset:(urn:li:dataPlatform:snowflake,...",space_role,None,None,JobTitle,None,None,"urn:li:dataset:(urn:li:dataPlatform:snowflake,...","[(JobTitle, 7.0)]",[],0,mismatch_actual_only
335,"urn:li:dataset:(urn:li:dataPlatform:snowflake,...",city,None,None,HomeAddress,Assigned HomeAddress because city can be a par...,None,"urn:li:dataset:(urn:li:dataPlatform:snowflake,...","[(HomeAddress, 8.0)]",[],0,mismatch_actual_only
336,"urn:li:dataset:(urn:li:dataPlatform:snowflake,...",state,None,None,HomeAddress,None,None,"urn:li:dataset:(urn:li:dataPlatform:snowflake,...","[(HomeAddress, 8.0)]",[],0,mismatch_actual_only
337,"urn:li:dataset:(urn:li:dataPlatform:snowflake,...",country,None,None,CountryOfResidence,None,None,"urn:li:dataset:(urn:li:dataPlatform:snowflake,...","[(CountryOfResidence, 8.0)]",[],0,mismatch_actual_only
338,"urn:li:dataset:(urn:li:dataPlatform:snowflake,...",region,None,None,HomeAddress,None,None,"urn:li:dataset:(urn:li:dataPlatform:snowflake,...","[(HomeAddress, 8.0)]",[],0,mismatch_actual_only
563,"urn:li:dataset:(urn:li:dataPlatform:redshift,t...",account_sid,None,None,Username,None,None,"urn:li:dataset:(urn:li:dataPlatform:redshift,t...","[(Username, 8.0)]",[],0,mismatch_actual_only
574,"urn:li:dataset:(urn:li:dataPlatform:snowflake,...",profile_id,Unique identifier of pet profile,"['M', 'M', 'M']",GenderIdentity,None,None,"urn:li:dataset:(urn:li:dataPlatform:snowflake,...","[(DeviceID, 7.0)]",[],0,mismatch_actual_only


In [405]:
# glossary_info
GlossaryInfo(glossary={'urn:li:glossaryTerm:PIITerms.Username': {'term_name': 'Username', 'term_description': 'A unique identifier chosen by the user for login purposes.', 'parent_node': {'parent_name': 'Information Types', 'parent_description': 'A set of terms related to Personally Identifiable Information commonly found in data warehouses for a company creating design tools.'}}, 'urn:li:glossaryTerm:PIITerms.SSN': {'term_name': 'SSN', 'term_description': 'Social Security Number of the user.', 'parent_node': {'parent_name': 'Information Types', 'parent_description': 'A set of terms related to Personally Identifiable Information commonly found in data warehouses for a company creating design tools.'}}, 'urn:li:glossaryTerm:PIITerms.PassportNumber': {'term_name': 'PassportNumber', 'term_description': "The user's passport number.", 'parent_node': {'parent_name': 'Information Types', 'parent_description': 'A set of terms related to Personally Identifiable Information commonly found in data warehouses for a company creating design tools.'}}, 'urn:li:glossaryTerm:PIITerms.CreditCardNumber': {'term_name': 'CreditCardNumber', 'term_description': "The user's credit card number.", 'parent_node': {'parent_name': 'Information Types', 'parent_description': 'A set of terms related to Personally Identifiable Information commonly found in data warehouses for a company creating design tools.'}}, 'urn:li:glossaryTerm:PIITerms.TwitterHandle': {'term_name': 'TwitterHandle', 'term_description': "The user's Twitter username.", 'parent_node': {'parent_name': 'Information Types', 'parent_description': 'A set of terms related to Personally Identifiable Information commonly found in data warehouses for a company creating design tools.'}}, 'urn:li:glossaryTerm:PIITerms.Nationality': {'term_name': 'Nationality', 'term_description': "The user's nationality.", 'parent_node': {'parent_name': 'Information Types', 'parent_description': 'A set of terms related to Personally Identifiable Information commonly found in data warehouses for a company creating design tools.'}}, 'urn:li:glossaryTerm:PIITerms.BankAccountNumber': {'term_name': 'BankAccountNumber', 'term_description': "The user's bank account number.", 'parent_node': {'parent_name': 'Information Types', 'parent_description': 'A set of terms related to Personally Identifiable Information commonly found in data warehouses for a company creating design tools.'}}, 'urn:li:glossaryTerm:PIITerms.BankRoutingNumber': {'term_name': 'BankRoutingNumber', 'term_description': "The bank routing number associated with the user's bank account.", 'parent_node': {'parent_name': 'Information Types', 'parent_description': 'A set of terms related to Personally Identifiable Information commonly found in data warehouses for a company creating design tools.'}}, 'urn:li:glossaryTerm:PIITerms.PhoneNumber': {'term_name': 'PhoneNumber', 'term_description': "The user's telephone number.", 'parent_node': {'parent_name': 'Information Types', 'parent_description': 'A set of terms related to Personally Identifiable Information commonly found in data warehouses for a company creating design tools.'}}, 'urn:li:glossaryTerm:PIITerms.FullName': {'term_name': 'FullName', 'term_description': "The user's full legal name.", 'parent_node': {'parent_name': 'Information Types', 'parent_description': 'A set of terms related to Personally Identifiable Information commonly found in data warehouses for a company creating design tools.'}}, 'urn:li:glossaryTerm:PIITerms.BillingAddress': {'term_name': 'BillingAddress', 'term_description': "The user's address used for billing purposes.", 'parent_node': {'parent_name': 'Information Types', 'parent_description': 'A set of terms related to Personally Identifiable Information commonly found in data warehouses for a company creating design tools.'}}, 'urn:li:glossaryTerm:PIITerms.CreditCardType': {'term_name': 'CreditCardType', 'term_description': 'The type of credit card used by the user.', 'parent_node': {'parent_name': 'Information Types', 'parent_description': 'A set of terms related to Personally Identifiable Information commonly found in data warehouses for a company creating design tools.'}}, 'urn:li:glossaryTerm:PIITerms.MedicalRecordNumber': {'term_name': 'MedicalRecordNumber', 'term_description': "The user's medical record number.", 'parent_node': {'parent_name': 'Information Types', 'parent_description': 'A set of terms related to Personally Identifiable Information commonly found in data warehouses for a company creating design tools.'}}, 'urn:li:glossaryTerm:PIITerms.WorkEmail': {'term_name': 'WorkEmail', 'term_description': "The user's work-related email address.", 'parent_node': {'parent_name': 'Information Types', 'parent_description': 'A set of terms related to Personally Identifiable Information commonly found in data warehouses for a company creating design tools.'}}, 'urn:li:glossaryTerm:PIITerms.JobTitle': {'term_name': 'JobTitle', 'term_description': 'The job title of the user.', 'parent_node': {'parent_name': 'Information Types', 'parent_description': 'A set of terms related to Personally Identifiable Information commonly found in data warehouses for a company creating design tools.'}}, 'urn:li:glossaryTerm:PIITerms.DeviceID': {'term_name': 'DeviceID', 'term_description': 'A unique identifier for a device used by the user.', 'parent_node': {'parent_name': 'Information Types', 'parent_description': 'A set of terms related to Personally Identifiable Information commonly found in data warehouses for a company creating design tools.'}}, 'urn:li:glossaryTerm:PIITerms.Age': {'term_name': 'Age', 'term_description': "A user's age.\n\n\xa0Examples Field Names:\n\n*   age\n    \n*   \\_age\\_\\_in\\_year\n    \n*   \\*\\_age\n    \n*   age\\_\\*", 'parent_node': {'parent_name': 'Information Types', 'parent_description': 'A set of terms related to Personally Identifiable Information commonly found in data warehouses for a company creating design tools.'}}, 'urn:li:glossaryTerm:PIITerms.FirstName': {'term_name': 'FirstName', 'term_description': "The user's first name.", 'parent_node': {'parent_name': 'Information Types', 'parent_description': 'A set of terms related to Personally Identifiable Information commonly found in data warehouses for a company creating design tools.'}}, 'urn:li:glossaryTerm:PIITerms.EmailAddress': {'term_name': 'EmailAddress', 'term_description': "The user's email address used for communication and as a login identifier.", 'parent_node': {'parent_name': 'Information Types', 'parent_description': 'A set of terms related to Personally Identifiable Information commonly found in data warehouses for a company creating design tools.'}}, 'urn:li:glossaryTerm:PIITerms.CountryOfResidence': {'term_name': 'CountryOfResidence', 'term_description': 'The country in which the user resides.', 'parent_node': {'parent_name': 'Information Types', 'parent_description': 'A set of terms related to Personally Identifiable Information commonly found in data warehouses for a company creating design tools.'}}, 'urn:li:glossaryTerm:PIITerms.HomeAddress': {'term_name': 'HomeAddress', 'term_description': "The user's residential address.", 'parent_node': {'parent_name': 'Information Types', 'parent_description': 'A set of terms related to Personally Identifiable Information commonly found in data warehouses for a company creating design tools.'}}, 'urn:li:glossaryTerm:PIITerms.LinkedInProfile': {'term_name': 'LinkedInProfile', 'term_description': "URL of the user's LinkedIn profile.", 'parent_node': {'parent_name': 'Information Types', 'parent_description': 'A set of terms related to Personally Identifiable Information commonly found in data warehouses for a company creating design tools.'}}, 'urn:li:glossaryTerm:PIITerms.BiometricData': {'term_name': 'BiometricData', 'term_description': 'Biometric data points such as fingerprints or retinal scans.', 'parent_node': {'parent_name': 'Information Types', 'parent_description': 'A set of terms related to Personally Identifiable Information commonly found in data warehouses for a company creating design tools.'}}, 'urn:li:glossaryTerm:PIITerms.TaxID': {'term_name': 'TaxID', 'term_description': 'The tax identification number of the user.', 'parent_node': {'parent_name': 'Information Types', 'parent_description': 'A set of terms related to Personally Identifiable Information commonly found in data warehouses for a company creating design tools.'}}, 'urn:li:glossaryTerm:PIITerms.EmploymentStatus': {'term_name': 'EmploymentStatus', 'term_description': "The user's employment status.", 'parent_node': {'parent_name': 'Information Types', 'parent_description': 'A set of terms related to Personally Identifiable Information commonly found in data warehouses for a company creating design tools.'}}, 'urn:li:glossaryTerm:PIITerms.MACAddress': {'term_name': 'MACAddress', 'term_description': "Media Access Control address of the user's device.", 'parent_node': {'parent_name': 'Information Types', 'parent_description': 'A set of terms related to Personally Identifiable Information commonly found in data warehouses for a company creating design tools.'}}, 'urn:li:glossaryTerm:PIITerms.LoginHistory': {'term_name': 'LoginHistory', 'term_description': 'Records of user logins including timestamps and IP addresses.', 'parent_node': {'parent_name': 'Information Types', 'parent_description': 'A set of terms related to Personally Identifiable Information commonly found in data warehouses for a company creating design tools.'}}, 'urn:li:glossaryTerm:PIITerms.FacebookProfile': {'term_name': 'FacebookProfile', 'term_description': "URL of the user's Facebook profile.", 'parent_node': {'parent_name': 'Information Types', 'parent_description': 'A set of terms related to Personally Identifiable Information commonly found in data warehouses for a company creating design tools.'}}, 'urn:li:glossaryTerm:PIITerms.BrowsingHistory': {'term_name': 'BrowsingHistory', 'term_description': 'Records of websites visited by the user.', 'parent_node': {'parent_name': 'Information Types', 'parent_description': 'A set of terms related to Personally Identifiable Information commonly found in data warehouses for a company creating design tools.'}}, 'urn:li:glossaryTerm:PIITerms.IPAddresses': {'term_name': 'IPAddresses', 'term_description': 'Internet Protocol addresses used by the user.', 'parent_node': {'parent_name': 'Information Types', 'parent_description': 'A set of terms related to Personally Identifiable Information commonly found in data warehouses for a company creating design tools.'}}, 'urn:li:glossaryTerm:PIITerms.LastName': {'term_name': 'LastName', 'term_description': "The user's last name.", 'parent_node': {'parent_name': 'Information Types', 'parent_description': 'A set of terms related to Personally Identifiable Information commonly found in data warehouses for a company creating design tools.'}}, 'urn:li:glossaryTerm:PIITerms.PasswordHash': {'term_name': 'PasswordHash', 'term_description': "A hashed representation of the user's password.", 'parent_node': {'parent_name': 'Information Types', 'parent_description': 'A set of terms related to Personally Identifiable Information commonly found in data warehouses for a company creating design tools.'}}, 'urn:li:glossaryTerm:PIITerms.MiddleName': {'term_name': 'MiddleName', 'term_description': "The user's middle name.", 'parent_node': {'parent_name': 'Information Types', 'parent_description': 'A set of terms related to Personally Identifiable Information commonly found in data warehouses for a company creating design tools.'}}, 'urn:li:glossaryTerm:PIITerms.ShippingAddress': {'term_name': 'ShippingAddress', 'term_description': 'Address where the user receives physical shipments.', 'parent_node': {'parent_name': 'Information Types', 'parent_description': 'A set of terms related to Personally Identifiable Information commonly found in data warehouses for a company creating design tools.'}}, 'urn:li:glossaryTerm:PIITerms.GenderIdentity': {'term_name': 'GenderIdentity', 'term_description': 'The gender with which the user identifies.', 'parent_node': {'parent_name': 'Information Types', 'parent_description': 'A set of terms related to Personally Identifiable Information commonly found in data warehouses for a company creating design tools.'}}, 'urn:li:glossaryTerm:PIITerms.InstagramHandle': {'term_name': 'InstagramHandle', 'term_description': "The user's Instagram username.", 'parent_node': {'parent_name': 'Information Types', 'parent_description': 'A set of terms related to Personally Identifiable Information commonly found in data warehouses for a company creating design tools.'}}, 'urn:li:glossaryTerm:PIITerms.CreditCardExpirationDate': {'term_name': 'CreditCardExpirationDate', 'term_description': "The expiration date of the user's credit card.", 'parent_node': {'parent_name': 'Information Types', 'parent_description': 'A set of terms related to Personally Identifiable Information commonly found in data warehouses for a company creating design tools.'}}, 'urn:li:glossaryTerm:PIITerms.DriverLicenseNumber': {'term_name': 'DriverLicenseNumber', 'term_description': "The user's driver's license number.", 'parent_node': {'parent_name': 'Information Types', 'parent_description': 'A set of terms related to Personally Identifiable Information commonly found in data warehouses for a company creating design tools.'}}, 'urn:li:glossaryTerm:PIITerms.BirthDate': {'term_name': 'BirthDate', 'term_description': "The user's date of birth.", 'parent_node': {'parent_name': 'Information Types', 'parent_description': 'A set of terms related to Personally Identifiable Information commonly found in data warehouses for a company creating design tools.'}}, 'urn:li:glossaryTerm:PIITerms.Cookies': {'term_name': 'Cookies', 'term_description': "Data files stored on the user's device used to track their website usage.", 'parent_node': {'parent_name': 'Information Types', 'parent_description': 'A set of terms related to Personally Identifiable Information commonly found in data warehouses for a company creating design tools.'}}, 'urn:li:glossaryTerm:PIITerms.HealthInsuranceID': {'term_name': 'HealthInsuranceID', 'term_description': "The user's health insurance ID number.", 'parent_node': {'parent_name': 'Information Types', 'parent_description': 'A set of terms related to Personally Identifiable Information commonly found in data warehouses for a company creating design tools.'}}, 'urn:li:glossaryTerm:PIITerms.WorkPhone': {'term_name': 'WorkPhone', 'term_description': "The user's work-related telephone number.", 'parent_node': {'parent_name': 'Information Types', 'parent_description': 'A set of terms related to Personally Identifiable Information commonly found in data warehouses for a company creating design tools.'}}, 'urn:li:glossaryTerm:PIITerms.EmployerName': {'term_name': 'EmployerName', 'term_description': "The name of the user's employer.", 'parent_node': {'parent_name': 'Information Types', 'parent_description': 'A set of terms related to Personally Identifiable Information commonly found in data warehouses for a company creating design tools.'}}, 'urn:li:glossaryTerm:PIITerms.CVV': {'term_name': 'CVV', 'term_description': "The card verification value of the user's credit card.", 'parent_node': {'parent_name': 'Information Types', 'parent_description': 'A set of terms related to Personally Identifiable Information commonly found in data warehouses for a company creating design tools.'}}})

NameError: name 'GlossaryInfo' is not defined

In [ ]:
# skipped_columns
# Current numbers
# assignment quality- few shot inference or restructuring
# reducing fake glossary term

In [ ]:
# sum(prediction_results_df.match)/len(prediction_results_df)

In [406]:
sample_urn = "urn:li:dataset:(urn:li:dataPlatform:snowflake,datahub_community.orbit_fivetran.member_in_workspace,PROD)"
column_infos_dict[sample_urn]
# table_infos_dict[sample_urn]

{'id': {'column_name': 'id',
  'metadata': {'nativeDataType': 'VARCHAR(256)', 'isPartOfKey': True},
  'upstream_lineages': []},
 'deleted_at': {'column_name': 'deleted_at',
  'metadata': {'nativeDataType': 'TIMESTAMP_TZ'},
  'upstream_lineages': []},
 'updated_at': {'column_name': 'updated_at',
  'metadata': {'nativeDataType': 'TIMESTAMP_TZ'},
  'upstream_lineages': []},
 'topics': {'column_name': 'topics',
  'metadata': {'nativeDataType': 'VARCHAR(256)'},
  'upstream_lineages': []},
 'created': {'column_name': 'created',
  'metadata': {'nativeDataType': 'BOOLEAN'},
  'upstream_lineages': []},
 'devto': {'column_name': 'devto',
  'metadata': {'nativeDataType': 'VARCHAR(256)'},
  'upstream_lineages': []},
 'url': {'column_name': 'url',
  'metadata': {'nativeDataType': 'VARCHAR(256)'},
  'upstream_lineages': []},
 'teammate': {'column_name': 'teammate',
  'metadata': {'nativeDataType': 'BOOLEAN'},
  'upstream_lineages': []},
 'github': {'column_name': 'github',
  'metadata': {'nativeData

In [ ]:
"""
Examples:
For a table with following information:
table info:
{{
    'name': 'member_in_workspace',
    'description': ''
}}

and provided list of Glossary terms:

Input:
Glossary_Terms:
['urn:li:glossaryTerm:PIITerms.GenderIdentity': {{'term_name': 'GenderIdentity', 'term_description': 'The gender with which the user identifies.', 'parent_node': {{'parent_name': 'Information Types', 'parent_description': 'A set of terms related to Personally Identifiable Information commonly found in data warehouses for a company creating design tools.'}}}},
'urn:li:glossaryTerm:PIITerms.FullName': {{'term_name': 'FullName', 'term_description': "The user's full legal name.", 'parent_node': {{'parent_name': 'Information Types', 'parent_description': 'A set of terms related to Personally Identifiable Information commonly found in data warehouses for a company creating design tools.'}}}},
'urn:li:glossaryTerm:PIITerms.MedicalRecordNumber': {{'term_name': 'MedicalRecordNumber', 'term_description': "The user's medical record number.", 'parent_node': {{'parent_name': 'Information Types', 'parent_description': 'A set of terms related to Personally Identifiable Information commonly found in data warehouses for a company creating design tools.'}}}},
...,]

Example1.
column_info:
'pronouns': {{
                'column_name': 'pronouns',
                'description': "",
                'sample_values': [],
                'datatype': 'VARCHAR(256)'
            
}}

assigned_terms:
[
    {{
          "urn": "urn:li:glossaryTerm:PIITerms.GenderIdentity",
          ""name"": ""GenderIdentity"",
          ""reasoning"": ""The pronouns of a person is gender specific information."",
          ""confidence_score"": 10.0,
    }},
]


Example2.
column_info:
'name': {{
            'column_name': 'name',
            'description': "",
            'sample_values': [],
            'datatype': 'VARCHAR(256)',
}}

assigned_term:
[
    {{
      ""urn"": ""urn:li:glossaryTerm:PIITerms.FullName"",
      ""name"": ""FullName"",
      ""reasoning"": ""The column contains the full names of members."",
      ""confidence_score"": 10.0,
    }}
]
  
Example3.

column_info:
'discourse': {{
                'column_name': 'discourse',
                'description': "Contains the discourse id of the user",
                'sample_values': [],
                'datatype': 'VARCHAR(256)',
}}
  
assigned_terms:
[]

"""

"""
"pronouns":{{
          "urn": "urn:li:glossaryTerm:PIITerms.GenderIdentity",
          ""name"": ""GenderIdentity"",
          ""reasoning"": ""The pronouns of a person is gender specific information."",
          ""confidence_score"": 10.0,
}}"""


"""
'name': {'column_name': 'name',
  'datatype': 'VARCHAR(256)',
  'sample_values': [],
  
  'upstream_lineages': []}
"""

"""
""name"": [
    {
      ""urn"": ""urn:li:glossaryTerm:PIITerms.FullName"",
      ""name"": ""FullName"",
      ""reasoning"": ""The 'name' column likely contains the full legal name of the Zendesk agent."",
      ""confidence_score"": 9.0,
      ""is_fake"": false
    }
  ]"""

In [ ]:
# very few columns have description, adding description could imp
# 